In [ ]:
!source ../miniconda3/bin/activate
!conda --version

'source' is not recognized as an internal or external command,
operable program or batch file.


conda 25.3.1


In [1]:
!nvidia-smi

Fri Jun 13 00:44:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.61                 Driver Version: 572.61         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   58C    P3             25W /   45W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------ --------------------- 1.0/2.3 MB 12.7 MB/s eta 0:00:01
   ---------------------- ----------------- 1.3/2.3 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 2.1/2.3 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 3.7 MB/s eta 0:00:00


In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.5.1
12.1
True


# Qwen2.5VL-3B

In [5]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("USING DEVICE: ",device)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",use_fast=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)
print("Successfully loaded Qwen2.5VL model")

USING DEVICE:  cuda


You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Successfully loaded Qwen2.5VL model


In [6]:
import time
messages = [
    {
        "role":"user",
        "content":[
            {
                "type":"image",
                "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
            },
            {
                "type":"text",
                "text":"Please describe this image"
            }
        ]
    }

]

start = time.time()

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
       generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)
print("Inference time:",time.time() - start,"seconds")

['The image shows a small, fluffy animal walking on a snowy surface. The animal has a thick, dense coat of fur that appears to be well-suited for cold weather. Its fur is primarily brown with some darker patches, and it has a bushy tail. The background features a chain-link fence and some birch trees, indicating an outdoor setting, possibly in a zoo or wildlife sanctuary. The snow on the ground suggests that it is winter.']
Inference time: 61.85375714302063 seconds


# SMolVLM2

In [9]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-256M-Video-Instruct")
model = AutoModelForImageTextToText.from_pretrained(
    "HuggingFaceTB/SmolVLM2-256M-Video-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
print("SmolVLM2 loaded succesfully")

SmolVLM2 loaded succesfully


In [13]:
import time

template = """
    "You are a visual analyst evaluating an image for signs of fire and the surrounding context. "
    "Do the following tasks:\n"
    "1: Summarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.\n"
    "2: Based on your summary, classify the fire situation: "
    "no fire(eg:, fire alarm, fire distinguisher,..), controlled fire (e.g., fireplace emitting, campfire, cooking, candles,..) or a dangerous/uncontrolled fire (e.g., curtains on fire, smoke on ceiling, couch on fire, bed sheet on fire,..)?\n"
    "Return only this JSON format:\n"
    "{ \"caption\": \"...\", \"label\": \"no fire\"|\"controlled fire\"|\"dangerous fire\" }"
"""

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image",
             "url": "https://assets.site-static.com/userFiles/2566/image/how-to-design-the-room-around-your-fireplace.jpg"},
            {"type": "text",
             "text": template}
        ]
    }
]

start_time = time.time()

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

output_ids = model.generate(**inputs, max_new_tokens=1024)
prompt_len = inputs["input_ids"].shape[1]
gen_tokens = output_ids[0, prompt_len:]
final_answer = processor.tokenizer.decode(
    gen_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
).strip()

print(final_answer)
print(f"Inference time: {(time.time() - start_time):.4f}s")

The fire situation is controlled.
Inference time: 2.2242s
